In [1]:
!pip install -q sentence-transformers rank_bm25 ranx pandas numpy codecarbon


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 384.6/384.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.4/467.4 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 M

In [2]:
from __future__ import annotations

import json
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

from rank_bm25 import BM25Okapi
from datasets import load_dataset

In [3]:
DATASET_NAME = "tmquan/phapdien-moj-gov-vn"
DATASET_CONFIG = "articles"
DATASET_SPLIT = "train"

QUESTION_PATH = Path("/kaggle/input/datasets/trnhuhuyhonggg/r2ai-raw-data/R2AIStage1DATA.json")

OUTPUT_PATH = Path("submission.json")
SAMPLE_OUTPUT_PATH = Path("sample_submission.json")

TOP_K = 4000
MAX_DOC_CHARS = 30000

TOKEN_RE = re.compile(r"[0-9a-zA-ZÀ-ỹĐđ]+", re.UNICODE)

ARTICLE_RE = re.compile(
    r"Điều\s+([0-9]+(?:\.[0-9]+)*)",
    re.IGNORECASE,
)

DOC_RE = re.compile(
    r"\b(Bộ luật|Luật|Nghị định|Thông tư liên tịch|Thông tư|Quyết định|"
    r"Nghị quyết|Pháp lệnh|Lệnh|Chỉ thị)\s+(?:số\s+)?([0-9][^\s,;)]+)",
    re.IGNORECASE,
)

DATE_BOUNDARY_RE = re.compile(
    r"\s+(?:ngày|của\s+Quốc hội|của\s+Chính phủ|của\s+Bộ|của\s+Thủ tướng|"
    r"của\s+Ủy ban|có\s+hiệu\s+lực)\b",
    re.IGNORECASE,
)

MOJIBAKE_MARKERS = ("Ã", "Â", "Æ", "Ä", "Å", "áº", "á»", "Ä‘")

VIETNAMESE_STOPWORDS = {
    "a", "ai", "anh", "ban", "bi", "biet", "bo", "cac", "cach", "can",
    "cho", "co", "con", "cua", "duoc", "gi", "hay", "hoac", "khi",
    "khong", "la", "lam", "mot", "nao", "neu", "nhu", "nhung", "o",
    "phai", "qua", "ra", "sao", "se", "thi", "the", "theo", "toi",
    "trong", "tu", "va", "ve", "voi",
}

In [4]:
def count_vietnamese_chars(text: str) -> int:
    return sum(
        1
        for char in text
        if "À" <= char <= "ỹ" or char in "Đđ"
    )


def maybe_fix_mojibake(value: Any) -> str:
    text = "" if value is None else str(value)

    if not any(marker in text for marker in MOJIBAKE_MARKERS):
        return text

    try:
        fixed = text.encode("latin1").decode("utf-8")
    except UnicodeError:
        return text

    if count_vietnamese_chars(fixed) > count_vietnamese_chars(text):
        return fixed

    return text


def clean_text(value: Any) -> str:
    return maybe_fix_mojibake(value).strip()


def strip_accents(text: str) -> str:
    normalized = unicodedata.normalize("NFD", text.lower())
    return "".join(
        char
        for char in normalized
        if unicodedata.category(char) != "Mn"
    )


def tokenize(text: str) -> list[str]:
    text = clean_text(text)
    text = strip_accents(text)

    tokens = TOKEN_RE.findall(text)

    return [
        token
        for token in tokens
        if len(token) > 1 and token not in VIETNAMESE_STOPWORDS
    ]

In [5]:
@dataclass
class LawArticle:
    id: str
    title: str
    content: str
    search_text: str
    doc_code: str
    doc_name: str
    article_label: str

In [6]:
def load_questions(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8-sig") as file:
        data = json.load(file)

    if isinstance(data, dict):
        data = (
            data.get("data")
            or data.get("questions")
            or data.get("items")
            or []
        )

    questions = []

    for item in data:
        questions.append(
            {
                "id": item.get("id"),
                "question": clean_text(item.get("question")),
            }
        )

    return questions

In [7]:
def normalize_doc_type(value: str) -> str:
    normalized = " ".join(value.split())

    mapping = {
        "bộ luật": "Bộ luật",
        "luật": "Luật",
        "nghị định": "Nghị định",
        "thông tư liên tịch": "Thông tư liên tịch",
        "thông tư": "Thông tư",
        "quyết định": "Quyết định",
        "nghị quyết": "Nghị quyết",
        "pháp lệnh": "Pháp lệnh",
        "lệnh": "Lệnh",
        "chỉ thị": "Chỉ thị",
    }

    return mapping.get(normalized.lower(), normalized)


def first_article_title_part(title: str) -> str:
    if not title:
        return ""

    return title.split(".", maxsplit=1)[0].strip()


def parse_legal_reference(
    source_note: str,
    link_notes: str,
    fallback_title: str,
) -> tuple[str, str, str]:
    reference_text = " ".join(
        part
        for part in [source_note, link_notes]
        if part
    )

    article_match = (
        ARTICLE_RE.search(reference_text)
        or ARTICLE_RE.search(fallback_title)
    )

    if article_match:
        article_label = f"Điều {article_match.group(1)}"
    else:
        article_label = first_article_title_part(fallback_title)

    doc_match = DOC_RE.search(reference_text)

    if not doc_match:
        return "", "Văn bản pháp luật", article_label

    doc_type = normalize_doc_type(doc_match.group(1))
    doc_code = doc_match.group(2).strip().strip(".,;)")

    summary_start = doc_match.end()
    summary_text = reference_text[summary_start:].strip(" .,-;()")
    summary_text = DATE_BOUNDARY_RE.split(summary_text, maxsplit=1)[0].strip(" .,-;()")

    doc_name = " ".join(
        part
        for part in [doc_type, doc_code, summary_text]
        if part
    )

    return doc_code, doc_name, article_label

# LOAD DATASET

In [8]:
dataset = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
    split=DATASET_SPLIT,
)

dataset

README.md: 0.00B [00:00, ?B/s]

articles-00000-of-00007.parquet:   0%|          | 0.00/6.07M [00:00<?, ?B/s]

articles-00001-of-00007.parquet:   0%|          | 0.00/6.56M [00:00<?, ?B/s]

articles-00002-of-00007.parquet:   0%|          | 0.00/5.84M [00:00<?, ?B/s]

articles-00003-of-00007.parquet:   0%|          | 0.00/6.60M [00:00<?, ?B/s]

articles-00004-of-00007.parquet:   0%|          | 0.00/6.20M [00:00<?, ?B/s]

articles-00005-of-00007.parquet:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

articles-00006-of-00007.parquet:   0%|          | 0.00/2.69M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/64464 [00:00<?, ? examples/s]

Dataset({
    features: ['subject_id', 'topic_id', 'topic_number', 'topic_title', 'subject_number', 'subject_title', 'article_anchor', 'article_title', 'chapter_title', 'source_note_text', 'source_links', 'related_note_text', 'content_text', 'content_char_len', 'content_word_count', 'source_url', 'scraped_at'],
    num_rows: 64464
})

In [9]:
def build_corpus(
    dataset: Iterable[dict[str, Any]],
    max_doc_chars: int = 30000,
) -> list[LawArticle]:
    corpus = []

    for index, row in enumerate(dataset):
        content = clean_text(row.get("content_text"))
        title = clean_text(row.get("article_title"))

        if not content and not title:
            continue

        source_note = clean_text(row.get("source_note_text"))

        source_links = row.get("source_links") or []
        link_notes = " ".join(
            clean_text(link.get("text"))
            for link in source_links
            if isinstance(link, dict)
        )

        doc_code, doc_name, article_label = parse_legal_reference(
            source_note=source_note,
            link_notes=link_notes,
            fallback_title=title,
        )

        if max_doc_chars > 0:
            searchable_content = content[:max_doc_chars]
        else:
            searchable_content = content

        search_text = "\n".join(
            part
            for part in [
                title,
                searchable_content,
                clean_text(row.get("subject_title")),
                clean_text(row.get("topic_title")),
                clean_text(row.get("chapter_title")),
                source_note,
            ]
            if part
        )

        article_id = clean_text(row.get("article_anchor")).lstrip("#")

        if not article_id:
            article_id = f"article-{index}"

        article = LawArticle(
            id=article_id,
            title=title,
            content=content,
            search_text=search_text,
            doc_code=doc_code,
            doc_name=doc_name,
            article_label=article_label,
        )

        corpus.append(article)

    return corpus

In [10]:
questions = load_questions(QUESTION_PATH)
corpus = build_corpus(dataset, max_doc_chars=MAX_DOC_CHARS)

print("Số câu hỏi:", len(questions))
print("Số điều luật:", len(corpus))

Số câu hỏi: 2000
Số điều luật: 64464


In [11]:
documents = [
    article.search_text
    for article in corpus
]

tokenized_documents = [
    tokenize(document)
    for document in documents
]

bm25 = BM25Okapi(tokenized_documents)

print("BM25 index ready")

BM25 index ready


In [12]:
def search_bm25(query: str, top_k: int = 50) -> list[tuple[int, float]]:
    query_tokens = tokenize(query)

    if not query_tokens:
        return []

    scores = bm25.get_scores(query_tokens)

    top_doc_ids = sorted(
        range(len(scores)),
        key=lambda doc_id: scores[doc_id],
        reverse=True,
    )[:top_k]

    results = []

    for doc_id in top_doc_ids:
        score = float(scores[doc_id])

        if score > 0:
            results.append((doc_id, score))

    return results

In [13]:
test_question = questions[0]["question"]

results = search_bm25(test_question, top_k=1)

print("Question:", test_question)
print()

for rank, (doc_id, score) in enumerate(results, start=1):
    article = corpus[doc_id]

    print(f"Rank {rank} | Score: {score:.4f}")
    print("Doc:", article.doc_name)
    print("Article:", article.article_label)
    print("Title:", article.title)
    print("Content:", article.content[:500])
    print("-" * 80)

Question: Các cơ sở ươm tạo và khu làm việc chung được hưởng những chính sách hỗ trợ nào về thuế và đất đai?

Rank 1 | Score: 53.0294
Doc: Luật 04/2017/QH14 có hiệu lực thi hành kể từ
Article: Điều 12
Title: Điều 12.3.LQ.12. Hỗ trợ công nghệ; hỗ trợ cơ sở ươm tạo, cơ sở kỹ thuật, khu làm việc chung
Content: 1. Nhà nước có chính sách hỗ trợ doanh nghiệp nhỏ và vừa nghiên cứu, đổi mới công nghệ, tiếp nhận, cải tiến, hoàn thiện, làm chủ công nghệ thông qua các hoạt động nghiên cứu, đào tạo, tư vấn, tìm kiếm, giải mã, chuyển giao công nghệ; xác lập, khai thác, quản lý, bảo vệ và phát triển tài sản trí tuệ của doanh nghiệp. 2. Bộ, cơ quan ngang Bộ, Ủy ban nhân dân cấp tỉnh thành lập cơ sở ươm tạo, cơ sở kỹ thuật, khu làm việc chung. Doanh nghiệp và tổ chức đầu tư, kinh doanh khác được thành lập cơ sở ươ
--------------------------------------------------------------------------------


# SINH ANSWER

In [14]:
def make_answer(top_articles: list[LawArticle]) -> str:
    if not top_articles:
        return "Chưa tìm thấy điều luật liên quan trong bộ dữ liệu."

    snippets = []

    for article in top_articles:
        content = article.content.replace("\n", " ").strip()

        if len(content) > 700:
            content = content[:700].rsplit(" ", maxsplit=1)[0] + "..."

        reference = " ".join(
            part
            for part in [
                article.doc_name,
                article.article_label,
                article.title,
            ]
            if part
        )

        snippets.append(f"{reference}: {content}")

    return "Các điều luật liên quan được truy xuất bằng BM25 gồm: " + " ".join(snippets)

In [15]:
def unique_relevant_docs(top_articles: list[LawArticle]) -> list[str]:
    seen = set()
    relevant_docs = []

    for article in top_articles:
        key = (article.doc_code, article.doc_name)

        if not article.doc_code:
            continue

        if key in seen:
            continue

        relevant_docs.append(f"{article.doc_code}|{article.doc_name}")
        seen.add(key)

    return relevant_docs


def relevant_articles(top_articles: list[LawArticle]) -> list[str]:
    rows = []

    for article in top_articles:
        if not article.doc_code:
            continue

        rows.append(
            f"{article.doc_code}|{article.doc_name}|{article.article_label}"
        )

    return rows

In [16]:
def truncate_text(text: str, max_chars: int = 1000) -> str:
    if not isinstance(text, str):
        text = str(text)

    text = text.strip()

    if len(text) <= max_chars:
        return text

    # Cắt không làm gãy từ
    return text[:max_chars].rsplit(" ", 1)[0].strip() + "..."

In [17]:
def build_submission(
    questions: list[dict[str, Any]],
    top_k: int = 3,
) -> list[dict[str, Any]]:
    submission = []

    for question in questions:
        results = search_bm25(
            query=question["question"],
            top_k=top_k,
        )

        top_articles = [
            corpus[doc_id]
            for doc_id, score in results
        ]

        item = {
            "id": question["id"],
            "question": question["question"],
            "answer": truncate_text(make_answer(top_articles), max_chars=1000),
            "relevant_docs": unique_relevant_docs(top_articles),
            "relevant_articles": relevant_articles(top_articles),
        }

        submission.append(item)

    return submission

In [18]:
submission = build_submission(
    questions=questions,
    top_k=TOP_K,
)

print("Số dòng submission:", len(submission))
submission[0]

def write_json(path: Path, data: Any) -> None:
    with path.open("w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )


write_json(OUTPUT_PATH, submission)

print("Wrote:", OUTPUT_PATH)



Số dòng submission: 2000
Wrote: submission.json


In [19]:
sample_submission = [
    {
        "id": 1,
        "question": "Nội dung câu hỏi pháp lý.",
        "answer": "Câu trả lời văn bản, có nhắc tới các điều luật liên quan.",
        "relevant_docs": [
            "04/2017/QH14|Luật 04/2017/QH14 Luật Hỗ trợ doanh nghiệp nhỏ và vừa"
        ],
        "relevant_articles": [
            "04/2017/QH14|Luật 04/2017/QH14 Luật Hỗ trợ doanh nghiệp nhỏ và vừa|Điều 4"
        ],
    }
]

write_json(SAMPLE_OUTPUT_PATH, sample_submission)

print("Wrote:", SAMPLE_OUTPUT_PATH)

Wrote: sample_submission.json


In [20]:
for item in submission[:3]:
    print("ID:", item["id"])
    print("Question:", item["question"])
    print("Relevant docs:", item["relevant_docs"])
    print("Relevant articles:", item["relevant_articles"])
    print("Answer:", item["answer"][:1000])
    print("=" * 100)

ID: 1
Question: Các cơ sở ươm tạo và khu làm việc chung được hưởng những chính sách hỗ trợ nào về thuế và đất đai?
Relevant docs: ['04/2017/QH14|Luật 04/2017/QH14 có hiệu lực thi hành kể từ', '10/2024/NĐ-CP|Nghị định 10/2024/NĐ-CP có hiệu lực thi hành kể từ', '07/2020/TT-BKHCN|Thông tư 07/2020/TT-BKHCN có hiệu lực thi hành kể từ', '21/2008/QH12|Luật 21/2008/QH12 có hiệu lực thi hành kể từ', '76/2018/NĐ-CP|Nghị định 76/2018/NĐ-CP có hiệu lực thi hành kể từ', '68/2014/QĐ-TTg|Quyết định 68/2014/QĐ-TTg có hiệu lực thi hành kể từ', '27/2013/TT-BKHCN|Thông tư 27/2013/TT-BKHCN có hiệu lực thi hành kể từ', '80/2021/NĐ-CP|Nghị định 80/2021/NĐ-CP có hiệu lực thi hành kể từ', '07/2020/TT-BKHCN|Thông tư 07/2020/TT-BKHCN Hướng dẫn việc thành lập cơ sở ươm tạo doanh nghiệp nhỏ và vừa, cơ sở kỹ thuật hỗ trợ doanh nghiệp nhỏ và vừa, khu làm việc chung hỗ trợ doanh nghiệp nhỏ và vừa khởi nghiệp sáng tạo', '38/2016/QĐ-TTg|Quyết định 38/2016/QĐ-TTg có hiệu lực thi hành kể từ', '61/2015/NĐ-CP|Nghị định 61